# pronostico mensual de inflacion

notebook para correr todos los meses. el analisis completo (eleccion de variables, modelos, tests) esta en `01_bcra_inflacion.ipynb`, aca queda solo lo necesario.

como se usa:
- **cuando el indec publica el dato del mes** (mitad de mes): correr todo. la seccion de seguimiento compara el dato con lo que publique, y el backtest suma ese mes.
- **cuando el bcra publica el rem** (4to dia habil del mes siguiente): correr todo. sale el pronostico del mes nuevo, se agrega a `data/pronosticos.csv` y se guarda el grafico en `figs/`.

no hay fechas escritas a mano: el mes a pronosticar es el primero que todavia no tiene dato de inflacion.

In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor

BAJAR=True   # True = baja todo de nuevo de las apis del bcra, False = usa lo que ya esta guardado en data/raw
HOY=pd.Timestamp.today().normalize()   # fecha de hoy, para pedir los datos hasta hoy

RAW=Path("../data/raw")   # datos tal cual vienen de las apis
PROC=Path("../data/processed")   # tablas armadas
FIGS=Path("../figs")   # graficos
REGISTRO=Path("../data/pronosticos.csv")   # historial de lo que publique cada mes
for carpeta in [RAW, PROC, FIGS]:
    carpeta.mkdir(parents=True, exist_ok=True)   # crea las carpetas si no existen

INICIO_MODELO="2019-01"   # desde aca hay rem de horizonte 0 todos los meses
INICIO_TEST="2022-01"   # desde aca arranca el backtest
ALPHA=0.2   # error tolerado del intervalo: 0.2 = intervalo del 80%
GAMMA=0.05   # que tan rapido se ajusta el aci
VENTANA=24   # cuantos meses de errores pasados uso para el ancho del intervalo
MIN_ERRORES=12   # minimo de errores para empezar a armar intervalos

dummies=[f"m_{i}" for i in range(1, 13)]
variables=["tc_var", "badlar_var", "res_var", "infl_l1", "infl_l2", "infl_l3", "rem"]+dummies   # las 19 variables del modelo

def rmse(e):
    return (e**2).mean()**0.5   # raiz del error cuadratico medio

## descarga

In [ ]:
URL_INFORMES="https://www.bcra.gob.ar/archivos/Pdfs/PublicacionesEstadisticas/informes/"   # carpeta del bcra donde estan los excel del rem

def descargar_serie(id_variable, desde, hasta):
    url=f"https://api.bcra.gob.ar/estadisticas/v4.0/monetarias/{id_variable}"
    limite=1000
    offset=0   # la api corta en 1000 filas, asi que pedimos por tandas
    filas=[]
    while True:
        r=requests.get(url, params={"desde": desde, "hasta": hasta, "limit": limite, "offset": offset})
        r.raise_for_status()   # a ver si salio bien
        detalle=r.json()["results"][0]["detalle"]
        filas.extend(detalle)
        if len(detalle)<limite:   # si vino menos que el limite, ya no hay mas
            break
        offset+=limite
    df=pd.DataFrame(filas)
    df["fecha"]=pd.to_datetime(df["fecha"])
    return df.sort_values("fecha").reset_index(drop=True)

def bajar_archivo(url, destino):
    r=requests.get(url)   # intento bajarlo
    if r.status_code!=200:   # 404 = todavia no lo publicaron
        return False
    with open(destino, "wb") as f:   # lo guardo tal cual viene
        f.write(r.content)
    return True

series={"tc_mayorista": 5, "badlar_nominal": 7, "reservas": 1, "inflacion_mensual": 27}   # nombre del archivo e id de la variable en la api del bcra

if BAJAR:
    for nombre, id_variable in series.items():
        d=descargar_serie(id_variable, "2017-01-01", HOY.strftime("%Y-%m-%d"))   # todo hasta hoy
        d.to_csv(RAW/f"bcra_{nombre}.csv", index=False)
        print(nombre, d.shape, d["fecha"].min().date(), d["fecha"].max().date())
    ok=bajar_archivo(URL_INFORMES+"relevamiento-expectativas-mercado-historico.xlsx", RAW/"rem_historico.xlsx")   # historico del rem
    print("rem historico:", "ok" if ok else "no se pudo bajar, uso el que ya estaba")
else:
    print("BAJAR=False: uso los archivos que ya estan en data/raw")

## tabla mensual

In [ ]:
def a_mensual(nombre, prefijo, como="pct"):
    d=pd.read_csv(RAW/f"bcra_{nombre}.csv", parse_dates=["fecha"])
    d["mes"]=d["fecha"].dt.to_period("M")
    m=d.groupby("mes")["valor"].agg(promedio="mean", ultimo="last").add_prefix(f"{prefijo}_")   # promedio y ultimo dato de cada mes
    if como=="pct":
        m[f"{prefijo}_var"]=m[f"{prefijo}_promedio"].pct_change()*100   # variacion % para niveles (dolar, reservas)
    else:
        m[f"{prefijo}_var"]=m[f"{prefijo}_promedio"].diff()   # variacion en puntos para tasas (badlar)
    return m

def rem_historico(ruta):
    rem=pd.read_excel(ruta, sheet_name="Base de Datos Completa", header=1)   # la fila 0 es el titulo
    sel=rem[rem["Variable"].str.contains("Precios minoristas", na=False) &
            rem["Variable"].str.contains("nivel general;", na=False) &   # ipc nivel general, definicion nacional
            (rem["Referencia"]=="var. % mensual")].copy()
    sel["mes_encuesta"]=pd.to_datetime(sel["Fecha de pronóstico"]).dt.to_period("M")
    sel["mes_periodo"]=pd.to_datetime(sel["Período"]).dt.to_period("M")
    sel=sel[sel["mes_encuesta"]==sel["mes_periodo"]]   # horizonte 0: lo que esperan para el mismo mes de la encuesta
    rem_mes=sel.set_index("mes_encuesta")["Mediana"].astype(float).rename("rem")
    assert rem_mes.index.is_unique   # un solo valor por mes, si no algo cambio en el excel
    return rem_mes

def rem_de_tablas(ruta, mes):
    t=pd.read_excel(ruta, sheet_name="Cuadros de resultados", header=None)   # el excel de tablas del informe del mes
    i=t.index[t[1]=="Precios minoristas (IPC nivel general-Nacional; INDEC)"][0]   # fila donde empieza el cuadro del ipc general
    j=i+2   # salteo el titulo y los nombres de las columnas
    while j<len(t) and pd.notna(t.loc[j,1]):   # recorro las filas del cuadro hasta la primera vacia
        periodo=pd.to_datetime(t.loc[j,1], errors="coerce")   # las filas de "prox. 12 meses" no son fechas y quedan NaT
        if pd.notna(periodo) and periodo.to_period("M")==mes and t.loc[j,2]=="var. % mensual":
            return float(t.loc[j,3])   # columna 3 = mediana
        j+=1
    return np.nan

base=a_mensual("tc_mayorista", "tc").join(a_mensual("badlar_nominal", "badlar", como="diff")).join(a_mensual("reservas", "res"))   # las tres series diarias pasadas a mensual

infl=pd.read_csv(RAW/"bcra_inflacion_mensual.csv", parse_dates=["fecha"])
infl["mes"]=infl["fecha"].dt.to_period("M")
tabla=base.join(infl.set_index("mes")["valor"].rename("inflacion"))   # la inflacion del bcra es lo que queremos predecir
tabla=tabla.loc["2017-06":]   # ene-may 2017 las fuentes no coincidian
tabla=tabla.join(rem_historico(RAW/"rem_historico.xlsx"))   # mediana del rem de horizonte 0

ultimo_dato=tabla["inflacion"].last_valid_index()   # ultimo mes que ya publico el indec
mes_obj=ultimo_dato+1   # el mes a pronosticar: el siguiente
print("ultimo dato del indec:", ultimo_dato, "| mes a pronosticar:", mes_obj)

if mes_obj not in tabla.index:
    tabla.loc[mes_obj]=np.nan   # por si todavia no hay datos diarios de ese mes

if pd.isna(tabla.loc[mes_obj, "rem"]):   # el historico se actualiza unos dias despues del informe, entonces busco el excel de tablas del mes
    ruta_tablas=RAW/f"rem_tablas_{mes_obj.year}_{mes_obj.month:02d}.xlsx"
    if BAJAR:
        bajar_archivo(URL_INFORMES+f"relevamiento-expectativas-mercado-tablas-{mes_obj.year}-{mes_obj.month:02d}.xlsx", ruta_tablas)
    if ruta_tablas.exists():
        tabla.loc[mes_obj, "rem"]=rem_de_tablas(ruta_tablas, mes_obj)
print("rem del mes a pronosticar:", tabla.loc[mes_obj, "rem"])

tabla["infl_l1"]=tabla["inflacion"].shift(1)   # rezagos de la inflacion
tabla["infl_l2"]=tabla["inflacion"].shift(2)
tabla["infl_l3"]=tabla["inflacion"].shift(3)
meses=pd.get_dummies(tabla.index.month, prefix="m", dtype=int)   # dummies de mes para la estacionalidad
meses.index=tabla.index
tabla=tabla.join(meses)

tabla.to_csv(PROC/"tabla_mensual.csv")
tabla[["tc_var", "badlar_var", "res_var", "infl_l1", "rem", "inflacion"]].tail(4)

## backtest

ventana expansiva desde 2022-01: cada mes se entrena solo con los meses anteriores. sirve para medir el error y para calibrar el intervalo.

In [ ]:
base_modelo=tabla.loc[INICIO_MODELO:ultimo_dato].dropna(subset=variables+["inflacion"])   # meses con todo completo y con dato real
X=base_modelo[variables]
y=base_modelo["inflacion"]
y_error=y-X["rem"]   # error del rem: lo que aprenden los modelos de correccion
meses_test=base_modelo.loc[INICIO_TEST:].index

armar_en=lambda: make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=[0.2, 0.5, 0.8], cv=TimeSeriesSplit(n_splits=5), max_iter=10000))   # elastic net con cv temporal
armar_rf=lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=3, max_features=0.5, random_state=0)   # random forest con hiperparametros fijos

def backtest(armar_modelo, sobre_rem=False):
    y_obj=y_error if sobre_rem else y   # que aprende: la inflacion o el error del rem
    preds=[]
    for mes in meses_test:
        modelo=armar_modelo()   # modelo nuevo en cada vuelta
        modelo.fit(X[X.index<mes], y_obj[y_obj.index<mes])   # solo meses anteriores
        p=modelo.predict(X.loc[[mes]])[0]
        if sobre_rem:
            p=p+X.loc[mes, "rem"]   # correccion + rem
        preds.append(p)
    return pd.Series(preds, index=meses_test)

resultados=base_modelo.loc[INICIO_TEST:, ["inflacion", "rem"]].copy()
resultados["en"]=backtest(armar_en)   # elastic net sobre la inflacion
resultados["en_rem"]=backtest(armar_en, sobre_rem=True)   # elastic net sobre el error del rem
resultados["rf_rem"]=backtest(armar_rf, sobre_rem=True)   # random forest sobre el error del rem
resultados["prom"]=resultados[["en", "en_rem", "rf_rem"]].mean(axis=1)   # lo que se publica

for desde in ["2022-01", "2024-05"]:
    d=resultados.loc[desde:]
    print(desde, len(d), "meses |", {c: round(rmse(d["inflacion"]-d[c]), 2) for c in ["en", "en_rem", "rf_rem", "prom", "rem"]})

In [ ]:
e_abs=(resultados["inflacion"]-resultados["prom"]).abs()   # error absoluto de cada mes
e_rel=e_abs/resultados["prom"]   # error relativo al nivel pronosticado

alpha_t=ALPHA   # aci: arranca en el valor objetivo
q_n=[]
for i, mes in enumerate(meses_test):
    pasados=e_rel.iloc[max(0, i-VENTANA):i]   # errores relativos de los meses anteriores, sin el actual
    if len(pasados)<MIN_ERRORES:
        q_n.append(np.nan)   # todavia no hay con que calibrar
    else:
        ancho=pasados.quantile(np.clip(1-alpha_t, 0, 1))*resultados.loc[mes, "prom"]   # percentil de errores relativos por el pronostico del mes
        q_n.append(ancho)
        fuera=e_abs.loc[mes]>ancho   # el real cayo afuera?
        alpha_t=alpha_t+GAMMA*(ALPHA-int(fuera))   # si fallo se ensancha, si acerto se achica un poco

resultados["q_n"]=pd.Series(q_n, index=meses_test)
resultados["inf_n"]=resultados["prom"]-resultados["q_n"]
resultados["sup_n"]=resultados["prom"]+resultados["q_n"]

v=resultados.dropna(subset=["q_n"])
for desde in ["2023-01", "2024-05"]:
    w=v.loc[desde:]
    cob=((w["inflacion"]>=w["inf_n"])&(w["inflacion"]<=w["sup_n"])).mean()
    print(desde, len(w), "meses | cobertura:", round(cob, 2), "| ancho medio:", round(2*w["q_n"].mean(), 2))
print("alpha_t al final del backtest:", round(alpha_t, 3))

## pronostico del mes nuevo

In [ ]:
fila=tabla.loc[[mes_obj], variables]   # la fila del mes a pronosticar con las 19 variables
faltan=fila.columns[fila.isna().iloc[0]].tolist()   # que variables estan vacias
mes_cerrado=mes_obj.end_time<HOY   # el mes tiene que haber terminado para tener los promedios completos
hay_pronostico=(len(faltan)==0) and mes_cerrado

if not mes_cerrado:
    print(f"{mes_obj} todavia no termino, los promedios del mes estarian incompletos")
elif faltan:
    print(f"no se puede pronosticar {mes_obj}, faltan: {faltan}")   # lo tipico: el rem todavia no salio
else:
    en=armar_en().fit(X, y)   # los tres modelos con todos los meses que tienen dato
    en_c=armar_en().fit(X, y_error)
    rf=armar_rf().fit(X, y_error)
    rem_obj=fila["rem"].iloc[0]
    p_en=en.predict(fila)[0]
    p_en_rem=en_c.predict(fila)[0]+rem_obj   # correccion + rem
    p_rf_rem=rf.predict(fila)[0]+rem_obj
    p_prom=(p_en+p_en_rem+p_rf_rem)/3

    rel=e_rel.iloc[-VENTANA:].quantile(np.clip(1-alpha_t, 0, 1))   # error relativo tipico de los ultimos 24 meses, con el alpha_t del final del backtest
    ancho=rel*p_prom   # mitad del intervalo en puntos de inflacion

    print(f"{mes_obj} | EN: {p_en:.2f} | EN_REM: {p_en_rem:.2f} | RF_REM: {p_rf_rem:.2f} | REM: {rem_obj}")
    print(f"pronostico: {p_prom:.2f}% | intervalo 80%: [{p_prom-ancho:.2f} ; {p_prom+ancho:.2f}]")

## registro

se agrega una fila por mes a `data/pronosticos.csv`. si el mes ya estaba no se pisa: lo publicado queda como se publico.

In [ ]:
if hay_pronostico:
    nueva=pd.DataFrame([{"mes": str(mes_obj), "fecha_pronostico": HOY.strftime("%Y-%m-%d"), "en": p_en, "en_rem": p_en_rem,
                         "rf_rem": p_rf_rem, "prom": p_prom, "inf": p_prom-ancho, "sup": p_prom+ancho, "rem": rem_obj, "alpha_t": alpha_t}])
    reg=pd.read_csv(REGISTRO) if REGISTRO.exists() else pd.DataFrame(columns=nueva.columns)   # lo que ya estaba registrado
    if str(mes_obj) in reg["mes"].astype(str).values:
        viejo=reg[reg["mes"].astype(str)==str(mes_obj)].iloc[0]
        print(f"{mes_obj} ya estaba registrado (pronostico {viejo['prom']:.2f}), no lo piso. hoy daria {p_prom:.2f}")
    else:
        reg=pd.concat([reg, nueva], ignore_index=True)   # agrego la fila nueva abajo
        reg.to_csv(REGISTRO, index=False)
        print(f"registrado {mes_obj}: {p_prom:.2f} [{p_prom-ancho:.2f} ; {p_prom+ancho:.2f}]")

## seguimiento

lo que publique contra lo que despues dijo el indec.

In [ ]:
if REGISTRO.exists():
    seg=pd.read_csv(REGISTRO)
    seg["real"]=seg["mes"].map(lambda m: tabla["inflacion"].get(pd.Period(m, "M"), np.nan))   # el dato del indec de ese mes (NaN si todavia no salio)
    seg["error"]=seg["real"]-seg["prom"]   # positivo = la inflacion fue mas alta que mi pronostico
    seg["error_rem"]=seg["real"]-seg["rem"]
    seg["adentro"]=(seg["real"]>=seg["inf"])&(seg["real"]<=seg["sup"])   # cayo dentro del intervalo?
    print(seg[["mes", "prom", "inf", "sup", "rem", "real", "error", "error_rem", "adentro"]].round(2).to_string(index=False))

## grafico

In [ ]:
w=resultados.dropna(subset=["q_n"]).loc["2024-05":]   # backtest desde mayo 2024
x=w.index.to_timestamp()
cob=((w["inflacion"]>=w["inf_n"])&(w["inflacion"]<=w["sup_n"])).mean()

fig, ax=plt.subplots(figsize=(11, 5.5), dpi=120)
ax.fill_between(x, w["inf_n"], w["sup_n"], color="#4C78A8", alpha=0.2, label="Intervalo 80% (ACI, normalizado)")
ax.plot(x, w["prom"], color="#4C78A8", marker="o", markersize=4, linewidth=2, label="Pronóstico (promedio simple de los 3 modelos)")
ax.plot(x, w["inflacion"], color="black", marker="o", markersize=4, linewidth=2, label="Inflación observada (INDEC)")

if hay_pronostico:   # el punto rojo del mes nuevo
    x_obj=mes_obj.to_timestamp()
    ax.plot([x[-1], x_obj], [w["prom"].iloc[-1], p_prom], color="#4C78A8", linestyle="--", linewidth=1.5)
    ax.errorbar(x_obj, p_prom, yerr=ancho, fmt="o", color="#E45756", capsize=6, markersize=8, linewidth=2, label=f"Pronóstico de {mes_obj}")
    ax.annotate(f"{p_prom:.2f}%  [{p_prom-ancho:.2f} ; {p_prom+ancho:.2f}]", (x_obj, p_prom), textcoords="offset points",
                xytext=(10, 45), ha="right", color="#E45756", fontsize=10, fontweight="bold")

ax.set_ylabel("Inflación mensual (%)")
ax.set_title("Nowcast de inflación mensual (backtest desde mayo del '24)", fontsize=13, fontweight="bold")
ax.grid(alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.text(0.01, 0.01, f"Backtest desde mayo de 2024 | RMSE: {rmse(w['inflacion']-w['prom']):.2f} (modelo) vs {rmse(w['inflacion']-w['rem']):.2f} (REM) | cobertura observada: {cob:.0%} (nominal: 80%)",
         fontsize=9, color="gray")
plt.tight_layout(rect=[0, 0.04, 1, 1])
if hay_pronostico:
    plt.savefig(FIGS/f"nowcast_{mes_obj}.png", dpi=200)   # un archivo por mes, ej nowcast_2026-11.png
plt.show()